In [12]:
import evidently
print(f"Versión instalada de Evidently: {evidently.__version__}")

Versión instalada de Evidently: 0.7.21


In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import re
from sklearn.metrics import f1_score, precision_score, accuracy_score

from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import DataSummaryPreset, DataDriftPreset
from evidently.ui.workspace import CloudWorkspace

print("Iniciando conexión con Evidently Cloud...")

# 1. Configuración de Evidently Cloud (Tus credenciales exactas)
TOKEN = "sk_prod.019e0c6a-0564-7203-9e44-2581ee9d9c99.Tuzed2cv1uwu6LcPapmvZLo-K2A0ATVZObaP1Z-5FvRNAFnTdF-XUlxXT_i-J-IM36sy-SpwYx7TYdqibiaAs5zIGeii2dYmT6frkj1FTJ6JNfw8FtKPL87KOqpO8bZQ"
ORG_ID = "019e0c67-8d3b-7865-9958-8a71c2859dac"

ws = CloudWorkspace(token=TOKEN, url="https://app.evidently.cloud")

# Buscamos el proyecto existente para no duplicarlo, si no existe lo crea
proyectos = ws.search_project("Proyecto_DISIA_Grupo_13")
if proyectos:
    project = proyectos[0]
    print("✅ Proyecto existente encontrado en la nube.")
else:
    project = ws.create_project(name="Proyecto_DISIA_Grupo_13", org_id=ORG_ID)
    project.description = "Monitorización de modelo de Tumores MRI"
    project.save()
    print("✨ Proyecto nuevo creado en la nube.")

# ==========================================================
# 2. CARGAR DATASET REAL (Referencia)
# ==========================================================
print("Cargando datos reales de las imágenes de Test...")
try:
    df_real = pd.read_csv("resultados_test_real.csv")
    print(f"Dataset real cargado con {len(df_real)} imágenes.")
except FileNotFoundError:
    print("❌ ERROR: No se encuentra 'resultados_test_real.csv'. Genera el archivo desde el cuaderno de entrenamiento primero.")
    raise

# ==========================================================
# 3. INYECCIÓN DE RUIDO - CHAOS ENGINEERING (Producción)
# ==========================================================
def inyectar_deriva(df_limpio):
    df_ruido = df_limpio.copy()
    
    # A. Deriva de Covariables: Simulamos un escáner MRI estropeado
    df_ruido['image_brightness'] = df_limpio['image_brightness'] * 0.6 + np.random.normal(0, 5, len(df_limpio))
    df_ruido['image_contrast'] = df_limpio['image_contrast'] - np.random.normal(20, 5, len(df_limpio))
    
    # B. Decaimiento del Modelo: El modelo baja su nivel de seguridad/confianza
    df_ruido['confidence'] = df_limpio['confidence'] * np.random.uniform(0.5, 0.8, len(df_limpio))
    df_ruido['confidence'] = np.clip(df_ruido['confidence'], 0, 1)
    
    # C. Concept Drift: Forzamos fallos asignando clases aleatorias al 30% de las imágenes
    mascara_fallos = np.random.rand(len(df_limpio)) < 0.30
    clases_posibles = df_limpio['target'].unique()
    df_ruido.loc[mascara_fallos, 'prediction'] = np.random.choice(clases_posibles, size=mascara_fallos.sum())
    
    return df_ruido

print("Aplicando ruido matemático para simular degradación en producción...")
df_produccion = inyectar_deriva(df_real)

# ==========================================================
# 4. MÉTRICAS MULTICLASE
# ==========================================================
print("\n" + "="*50)
print("📊 COMPARATIVA DE RENDIMIENTO (REAL vs PRODUCCIÓN)")
print("--- DATASET ORIGINAL (Referencia / Sano) ---")
print(f"Accuracy:  {accuracy_score(df_real['target'], df_real['prediction']):.4f}")
print(f"F1 Score:  {f1_score(df_real['target'], df_real['prediction'], average='weighted'):.4f}")

print("\n--- DATASET CON RUIDO (Producción / Escáner Roto) ---")
print(f"Accuracy:  {accuracy_score(df_produccion['target'], df_produccion['prediction']):.4f}")
print(f"F1 Score:  {f1_score(df_produccion['target'], df_produccion['prediction'], average='weighted'):.4f}")
print("="*50 + "\n")

# ==========================================================
# 5. SUBIDA A EVIDENTLY CLOUD
# ==========================================================
print("Preparando esquemas de Evidently...")
schema = DataDefinition(
    numerical_columns=["confidence", "image_brightness", "image_contrast"],
    categorical_columns=["target", "prediction"]
)

# El dataset 'eval_data_1' es el de producción (el que tiene ruido)
eval_data_1 = Dataset.from_pandas(df_produccion, data_definition=schema)

# El dataset 'eval_data_2' es el de referencia (las imágenes sanas del test)
eval_data_2 = Dataset.from_pandas(df_real, data_definition=schema)

print("Ejecutando evaluación de Data Drift...")
report = Report([
    DataSummaryPreset(),
    DataDriftPreset()
], include_tests="True")

my_eval = report.run(eval_data_1, eval_data_2)

print("Subiendo métricas al servidor de Evidently Cloud...")
ws.add_run(project.id, my_eval, include_data=False)

# ==========================================================
# 6. EVALUACIÓN DE DERIVA Y DISPARO DE WEBHOOK AUTOMÁTICO
# ==========================================================
print("Analizando los resultados internos de la evaluación para el Webhook...")

# 1. Obtenemos el diccionario con los resultados
try:
    report_dict = my_eval.as_dict()
except AttributeError:
    report_dict = my_eval.dict()

detecciones_drift = []
columnas_con_deriva = 0

# 2. BÚSQUEDA PRECISA (Basada en tu JSON)
# Recorremos la lista de "tests" buscando los que analizan la deriva y han fallado (FAIL)
for test in report_dict.get("tests", []):
    test_name = test.get("name", "")
    
    # Si el test es sobre Data Drift...
    if test_name.startswith("Value Drift for column"):
        # Comprobamos si el estado es FAIL (usamos str() por si viene como objeto TestStatus)
        estado = str(test.get("status", ""))
        
        if "FAIL" in estado:
            col_name = test.get("metric_config", {}).get("params", {}).get("column", "Desconocida")
            descripcion = test.get("description", "Sin detalles")
            
            detecciones_drift.append({
                "columna": col_name,
                "estado": "¡DERIVA DETECTADA!",
                "detalles": descripcion
            })
            columnas_con_deriva += 1

# 3. GENERACIÓN DEL CSV PARA LA MEMORIA
if detecciones_drift:
    df_detecciones = pd.DataFrame(detecciones_drift)
    df_detecciones.to_csv("detecciones_data_drift_real.csv", index=False)
    print(f"\n✅ ¡ÉXITO! CSV generado con {columnas_con_deriva} alertas.")
    display(df_detecciones)  # Muestra la tabla bonita en Jupyter
else:
    print("\n✅ Los datos están sanos. No se han detectado tests en estado FAIL.")

# 4. DISPARO DEL WEBHOOK
if columnas_con_deriva > 0:
    print(f"\n⚠️ ¡ALERTA CRÍTICA! Se detectó deriva en {columnas_con_deriva} columnas.")
    print("🚀 Disparando webhook al sistema de reentrenamiento...")
    
    WEBHOOK_URL = "http://localhost:9091/trigger-retrain"
    
    # Le pasamos al webhook exactamente qué columnas han fallado
    lista_columnas = [d['columna'] for d in detecciones_drift]
    payload = {
        "status": "firing",
        "reason": "data_drift",
        "detalles": f"Deriva detectada en el escáner MRI. Columnas afectadas: {', '.join(lista_columnas)}"
    }
    
    try:
        respuesta = requests.post(WEBHOOK_URL, json=payload, timeout=5)
        if respuesta.status_code == 200:
            print("✅ Webhook notificado con éxito. El script bash ya está trabajando.")
        else:
            print(f"❌ Fallo al notificar al webhook. HTTP: {respuesta.status_code}")
    except requests.exceptions.ConnectionError:
        print("❌ Error: No se pudo contactar con el Webhook en el puerto 9000. ¿Está encendido tu contenedor 'retrain-trigger'?")

print("✅ ¡ÉXITO TOTAL! Entra en https://app.evidently.cloud para ver el desastre simulado.")

Iniciando conexión con Evidently Cloud...
✅ Proyecto existente encontrado en la nube.
Cargando datos reales de las imágenes de Test...
Dataset real cargado con 624 imágenes.
Aplicando ruido matemático para simular degradación en producción...

📊 COMPARATIVA DE RENDIMIENTO (REAL vs PRODUCCIÓN)
--- DATASET ORIGINAL (Referencia / Sano) ---
Accuracy:  0.8798
F1 Score:  0.8792

--- DATASET CON RUIDO (Producción / Escáner Roto) ---
Accuracy:  0.7003
F1 Score:  0.6988

Preparando esquemas de Evidently...
Ejecutando evaluación de Data Drift...
Subiendo métricas al servidor de Evidently Cloud...
Analizando los resultados internos de la evaluación para el Webhook...

✅ ¡ÉXITO! CSV generado con 3 alertas.


,columna,estado,detalles
0,confidence,¡DERIVA DETECTADA!,Drift score is 0.00. The drift detection metho...
1,image_brightness,¡DERIVA DETECTADA!,Drift score is 0.00. The drift detection metho...
2,image_contrast,¡DERIVA DETECTADA!,Drift score is 0.00. The drift detection metho...



⚠️ ¡ALERTA CRÍTICA! Se detectó deriva en 3 columnas.
🚀 Disparando webhook al sistema de reentrenamiento...
❌ Error: No se pudo contactar con el Webhook en el puerto 9000. ¿Está encendido tu contenedor 'retrain-trigger'?
✅ ¡ÉXITO TOTAL! Entra en https://app.evidently.cloud para ver el desastre simulado.
